In [1]:
import sys
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

In [2]:
import sys
project_root = "/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [3]:


# ── Config ────────────────────────────────────────────────────────────────
BASE_CACHE = Path('/data/krosen5/cache/transformers/balanced_8')
MODELS_DIR = BASE_CACHE / 'saved_models'
OUTPUT_DIR = BASE_CACHE / 'error_analysis_v2'
TARGET     = 'DOMESTIC_STAY_AWAY_ORDER_IMPOSED'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Helpers ───────────────────────────────────────────────────────────────
def load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

def get_proc_text(proc):
    for k in ('text', 'proceeding_text', 'mag_text', 'cdc_text', 'notes'):
        v = proc.get(k, '')
        if v and str(v).strip():
            return str(v).strip()
    return '[no text]'

def get_proc_meta(proc):
    return {
        'date':      str(proc.get('date', '')),
        'proc_type': proc.get('type', ''),
        'case_id':   proc.get('case_id', '') or proc.get('docket_id', ''),
        'def_id':    proc.get('defendant_id', ''),
    }

def format_step(i, proc):
    meta = get_proc_meta(proc)
    text = get_proc_text(proc)
    return f"[Step {i} | {meta['date']} | {meta['proc_type']}]\n{text}"

def build_context_text(sequence, up_to_exclusive):
    """
    Text of proceedings the model SAW when predicting step `up_to_exclusive`.
    = proceedings 0 .. up_to_exclusive-1
    Given the architecture: to predict step t, model sees steps 0..t-1.
    """
    if up_to_exclusive == 0:
        return '[No prior proceedings — model predicted Step 0 from initial state only (case metadata, defendant demographics, charge type)]'
    parts = [format_step(i, sequence[i]) for i in range(up_to_exclusive)]
    return '\n\n---\n\n'.join(parts)

def build_full_text(sequence):
    parts = [format_step(i, p) for i, p in enumerate(sequence)]
    return '\n\n---\n\n'.join(parts)

# ── Styling ───────────────────────────────────────────────────────────────
PALETTE = {
    'FP':          ('8B0000', 'FDECEA'),
    'FN':          ('7B4F00', 'FFF8E1'),
    'Qualitative': ('4A235A', 'F5EEF8'),
}

WIDE = {
    'input_context_text':    85,   # what the model SAW  (0..t-1)
    'target_step_text':      80,   # text AT step t      (what was predicted)
    'all_proceedings_text':  90,
    'architecture_note':     70,
    'true_labels':           50,
    'predicted_labels':      50,
    'case_fp_labels':        50,
    'case_fn_labels':        50,
    'error_type':            55,
}

def write_wb(rows, path, key, title):
    if not rows:
        print(f"  ⚠️  {key}: no rows — skipping")
        return
    hdr, alt = PALETTE[key]
    wb = Workbook()
    ws = wb.active
    ws.title = key
    cols = list(rows[0].keys())

    ws.merge_cells(f'A1:{get_column_letter(len(cols))}1')
    ws['A1'] = title
    ws['A1'].font      = Font(name='Arial', bold=True, size=13, color=hdr)
    ws['A1'].alignment = Alignment(horizontal='center')
    ws.row_dimensions[1].height = 24
    ws.append([])

    ws.append(cols)
    hr = ws.max_row
    for c, col in enumerate(cols, 1):
        cell = ws.cell(row=hr, column=c)
        cell.font      = Font(name='Arial', bold=True, color='FFFFFF', size=10)
        cell.fill      = PatternFill('solid', start_color=hdr)
        cell.alignment = Alignment(horizontal='center')
    ws.freeze_panes = ws.cell(row=hr + 1, column=1)

    for i, row in enumerate(rows):
        ws.append([row.get(c, '') for c in cols])
        dr   = ws.max_row
        fill = PatternFill('solid', start_color=alt) if i % 2 == 1 else None
        for c, col in enumerate(cols, 1):
            cell = ws.cell(row=dr, column=c)
            cell.font = Font(name='Arial', size=9)
            if fill:
                cell.fill = fill
            if col in WIDE:
                cell.alignment = Alignment(wrap_text=True, vertical='top')
        ws.row_dimensions[dr].height = 100

    for c, col in enumerate(cols, 1):
        ws.column_dimensions[get_column_letter(c)].width = WIDE.get(col, 16)

    wb.save(path)
    print(f"  ✅ {key:14s} {len(rows):4d} rows  →  {path.name}")


# ── Load ──────────────────────────────────────────────────────────────────
print('=' * 70)
print(f'  Step-Level Error Analysis (architecture-aware): {TARGET}')
print(__doc__)
print('=' * 70)

mr            = load_pickle(MODELS_DIR / 'model_results.pkl')
fold_results  = mr['cv_training_results']['fold_results']
case_lvl_dict = mr['case_level_dict']
misclass_dict = mr['confident_misclass_results_per_fold']

fp_rows, fn_rows = [], []

for fold_idx, fold_data in enumerate(fold_results):
    val_data   = fold_data['val_data']   # list of (sequence, labels_raw)
    case_dicts = case_lvl_dict[fold_idx]

    # ── Pick misclassification DataFrame ──────────────────────────────────
    mc_data  = misclass_dict[fold_idx]
    mc_df    = None
    used_key = None
    for df_key in ('interpreted_mistakes', 'mistakes_report'):
        df = mc_data.get(df_key)
        if df is not None and isinstance(df, pd.DataFrame) and not df.empty:
            mc_df    = df.copy()
            used_key = df_key
            break

    if mc_df is None or mc_df.empty:
        print(f'  Fold {fold_idx+1}: no misclass data — skipping')
        continue

    print(f'\n── Fold {fold_idx+1} [{used_key}] | val={len(val_data)} | mistakes={len(mc_df)} ──')
    print(f'   Columns: {list(mc_df.columns)}')

    # ── Detect column names dynamically ───────────────────────────────────
    text_col     = next((c for c in ('proceeding_text', 'text', 'mag_text', 'cdc_text')
                         if c in mc_df.columns), None)
    case_idx_col = next((c for c in ('case_idx', 'val_idx', 'sequence_idx')
                         if c in mc_df.columns), None)
    # step_idx_col: the step t being predicted (labels_binary[t])
    step_idx_col = next((c for c in ('step_idx', 'proceeding_idx',
                                     'target_proceeding_idx', 'proc_idx')
                         if c in mc_df.columns), None)
    conf_col     = next((c for c in ('confidence', 'prob', 'prediction_prob')
                         if c in mc_df.columns), None)
    wrong_col    = next((c for c in ('wrong_label', 'predicted_label', 'label')
                         if c in mc_df.columns), None)
    true_col     = next((c for c in ('true_labels', 'true_label', 'true')
                         if c in mc_df.columns), None)

    # FP: model predicted TARGET at step t, but TARGET not in step t's true labels
    fp_mask = pd.Series(False, index=mc_df.index)
    if wrong_col:
        fp_mask = mc_df[wrong_col].astype(str) == TARGET

    # FN: TARGET was true at step t, but model predicted something else
    fn_mask = pd.Series(False, index=mc_df.index)
    if true_col:
        fn_mask = mc_df[true_col].astype(str).str.contains(TARGET, na=False)
        if wrong_col:
            fn_mask = fn_mask & (mc_df[wrong_col].astype(str) != TARGET)

    fp_steps = mc_df[fp_mask].copy()
    fn_steps = mc_df[fn_mask].copy()
    print(f'   FPs for {TARGET}: {len(fp_steps)}')
    print(f'   FNs for {TARGET}: {len(fn_steps)}')

    def enrich(step_df, is_fp):
        out = []
        for _, srow in step_df.iterrows():
            r = dict(srow)

            # ── Step t: the prediction target ────────────────────────────
            t = r.get(step_idx_col, None) if step_idx_col else None
            if t is not None:
                try:
                    t = int(t)
                except (ValueError, TypeError):
                    t = None

            confidence = r.get(conf_col,  float('nan')) if conf_col  else float('nan')
            wrong_pred = r.get(wrong_col, '')           if wrong_col else ''
            true_lbl   = r.get(true_col,  '')           if true_col  else ''

            # ── Link to val_data via case_idx ─────────────────────────────
            c_idx = r.get(case_idx_col, None) if case_idx_col else None
            if c_idx is not None:
                try:
                    c_idx = int(c_idx)
                except (ValueError, TypeError):
                    c_idx = None

            sequence       = []
            full_text      = '[case not linked]'
            case_f1        = float('nan')
            case_true      = str(true_lbl)
            case_pred      = ''
            case_fps       = ''
            case_fns       = ''
            case_id        = ''
            num_steps      = '?'

            if c_idx is not None and c_idx < len(val_data):
                seq_item  = val_data[c_idx]
                sequence  = seq_item[0] if isinstance(seq_item, (tuple, list)) else seq_item
                full_text = build_full_text(sequence)
                num_steps = len(sequence)
                if sequence:
                    case_id = get_proc_meta(sequence[0]).get('case_id', '')

                matching = [cd for cd in case_dicts if cd.get('case_idx') == c_idx]
                if matching:
                    cd        = matching[0]
                    case_f1   = round(float(cd.get('f1',  float('nan'))), 4)
                    case_true = ' | '.join(cd.get('true_labels',     []))
                    case_pred = ' | '.join(cd.get('pred_labels',     []))
                    case_fps  = ' | '.join(cd.get('false_positives', []))
                    case_fns  = ' | '.join(cd.get('false_negatives', []))

            # ── Architecture-aware text columns ───────────────────────────
            #
            # The model predicts step t's labels from proceedings 0..t-1.
            # "Step t-1 → Step t"
            #
            # input_context_text : proceedings 0..t-1 — what the model SAW
            # target_step_text   : text at step t    — what was being predicted

            if t is not None and sequence:
                input_context_text = build_context_text(sequence, up_to_exclusive=t)
                target_step_text   = (format_step(t, sequence[t])
                                      if t < len(sequence)
                                      else '[step index out of range]')
            elif text_col:
                # Fallback: misclass df has the step text directly
                input_context_text = '[step index unavailable — see target_step_text]'
                target_step_text   = str(r.get(text_col, '[no text]'))
            else:
                input_context_text = '[unavailable]'
                target_step_text   = '[unavailable]'

            # Step description (matches notebook format)
            if t is not None:
                if t == 0:
                    step_desc = 'Initial → Step 0'
                else:
                    step_desc = f'Step {t-1} → Step {t}'
            else:
                step_desc = '?'

            # Architecture note tailored to this error
            if is_fp:
                arch_note = (
                    f"FP at {step_desc}: model saw proceedings 0..{t-1 if t else '?'} "
                    f"and WRONGLY predicted {TARGET} at step {t}. "
                    f"input_context_text shows WHAT THE MODEL SAW that triggered the false prediction. "
                    f"target_step_text confirms no stay-away at step {t}."
                )
            else:
                if t == 0:
                    arch_note = (
                        f"FN at {step_desc}: {TARGET} was at Step 0 (first appearance) "
                        f"but model had NO prior proceedings to predict from — only "
                        f"initial case metadata (charge type, defendant demographics). "
                        f"input_context_text is EMPTY. target_step_text has the stay-away language."
                    )
                else:
                    arch_note = (
                        f"FN at {step_desc}: {TARGET} was at step {t} but model, "
                        f"seeing only proceedings 0..{t-1}, missed it. "
                        f"input_context_text shows what the model saw. "
                        f"target_step_text has the stay-away language the model could not yet see."
                    )

            out.append({
                # ── Error identity ─────────────────────────────────────
                'error_type':         ('FALSE POSITIVE — model predicted stay-away; none imposed'
                                       if is_fp else
                                       'FALSE NEGATIVE — stay-away imposed; model missed it'),
                'fold':               fold_idx + 1,
                'case_idx':           c_idx,
                'case_id':            case_id,
                'num_proceedings':    num_steps,
                'prediction_target':  step_desc,
                'model_confidence':   round(float(confidence), 4)
                                      if not pd.isna(confidence) else '',
                'wrong_prediction':   wrong_pred,
                # ── Architecture note ──────────────────────────────────
                'architecture_note':  arch_note,
                # ── KEY TEXT COLUMNS ───────────────────────────────────
                # input_context_text: proceedings 0..t-1 the model SAW
                # (empty string = Step 0 prediction from cold start)
                'input_context_text': input_context_text,
                # target_step_text: text AT step t (the prediction target)
                # FP → confirms no stay-away here despite model predicting it
                # FN → contains the stay-away language model had to predict blindly
                'target_step_text':   target_step_text,
                # ── Case-level context ─────────────────────────────────
                'case_f1':            case_f1,
                'case_true_labels':   case_true,
                'case_pred_labels':   case_pred,
                'case_fp_labels':     case_fps,
                'case_fn_labels':     case_fns,
                # ── Full history ───────────────────────────────────────
                'all_proceedings_text': full_text,
            })
        return out

    fp_rows.extend(enrich(fp_steps, is_fp=True))
    fn_rows.extend(enrich(fn_steps, is_fp=False))

# ── Diagnostic ────────────────────────────────────────────────────────────
if not fp_rows and not fn_rows:
    print('\n⚠️  Both lists empty — column diagnostic:')
    for fi in range(5):
        mc_data = misclass_dict[fi]
        for key in ('interpreted_mistakes', 'mistakes_report'):
            df = mc_data.get(key)
            if df is not None and isinstance(df, pd.DataFrame) and not df.empty:
                print(f'\n  Fold {fi+1} [{key}] columns: {list(df.columns)}')
                print(f'  Row 0: {df.iloc[0].to_dict()}')
                break
    sys.exit(0)

# ── Sort worst cases first ────────────────────────────────────────────────
fp_rows.sort(key=lambda r: float(r.get('case_f1') or 1.0))
fn_rows.sort(key=lambda r: float(r.get('case_f1') or 1.0))
qual_rows = sorted(fp_rows + fn_rows, key=lambda r: float(r.get('case_f1') or 1.0))

# ── Write ─────────────────────────────────────────────────────────────────
print(f'\n{"─"*70}')
write_wb(fp_rows,
         OUTPUT_DIR / 'stay_away_FP_step_level.xlsx', 'FP',
         f'{TARGET} — False Positives | input_context_text = what model SAW (triggered FP)')
write_wb(fn_rows,
         OUTPUT_DIR / 'stay_away_FN_step_level.xlsx', 'FN',
         f'{TARGET} — False Negatives | target_step_text = where stay-away is; input often empty (Step 0)')
write_wb(qual_rows,
         OUTPUT_DIR / 'stay_away_qualitative_step_level.xlsx', 'Qualitative',
         f'All Errors — {TARGET} | Sorted worst F1 first | See architecture_note column')

print(f'\n{"="*70}')
print(f'  Output : {OUTPUT_DIR}')
print(f'  FP rows: {len(fp_rows)}  |  FN rows: {len(fn_rows)}')
print(f"""
  KEY COLUMNS:
    input_context_text  — proceedings 0..t-1 the model SAW before predicting step t
                          (EMPTY for Step 0 FNs: model had no prior text to go on)
    target_step_text    — text AT step t (the prediction target)
                          FP → confirms no stay-away despite model prediction
                          FN → contains the stay-away language
    architecture_note   — plain-English description of what happened at this step
    prediction_target   — "Step t-1 → Step t" format matching notebook output
""")
print('=' * 70)

  Step-Level Error Analysis (architecture-aware): DOMESTIC_STAY_AWAY_ORDER_IMPOSED
Automatically created module for IPython interactive environment

── Fold 1 [interpreted_mistakes] | val=507 | mistakes=679 ──
   Columns: ['case_id', 'case_idx', 'proceeding_id', 'step_idx', 'true_labels', 'predicted_labels', 'confidences', 'wrong_label', 'wrong_label_confidence', 'sequence_length', 'num_true_labels', '_sequence', '_teacher_forced_sequence', '_true_binary', 'top_features', 'top_feature_scores', 'component_importance']
   FPs for DOMESTIC_STAY_AWAY_ORDER_IMPOSED: 14
   FNs for DOMESTIC_STAY_AWAY_ORDER_IMPOSED: 24

── Fold 2 [interpreted_mistakes] | val=507 | mistakes=685 ──
   Columns: ['case_id', 'case_idx', 'proceeding_id', 'step_idx', 'true_labels', 'predicted_labels', 'confidences', 'wrong_label', 'wrong_label_confidence', 'sequence_length', 'num_true_labels', '_sequence', '_teacher_forced_sequence', '_true_binary', 'top_features', 'top_feature_scores', 'component_importance']
   FPs

In [4]:
from collections import Counter
for _label, _rows in [("FP", fp_rows), ("FN", fn_rows)]:
    if not _rows:
        print(f"  {_label}: no rows")
        continue
    step_counts = Counter(r.get("prediction_target", "?") for r in _rows)
    print(f"\n  {_label} ({len(_rows)} total):")
    for step, count in sorted(step_counts.items(), key=lambda x: x[1], reverse=True):
        pct = 100 * count / len(_rows)
        flag = "  <- cold start (no prior text)" if "Initial" in step else ""
        print(f"    {step}: {count} ({pct:.0f}%){flag}")


  FP (80 total):
    Initial → Step 0: 80 (100%)  <- cold start (no prior text)

  FN (93 total):
    Initial → Step 0: 90 (97%)  <- cold start (no prior text)
    Step 0 → Step 1: 2 (2%)
    Step 3 → Step 4: 1 (1%)
